In [ ]:
import weaviate
from weaviate.classes.config  import Configure,Property, DataType
import requests, json
import base64

client = weaviate.connect_to_local(
    host="172.17.0.4",  
    port=8080,
    grpc_port=50051,
)

#Only need to run once to test create the collection

collections = client.collections.delete("Grounded_nomic")
print(collections)


questions = client.collections.create(
    name="Grounded_nomic",
    vector_config=Configure.Vectors.text2vec_ollama(  # Configure the Ollama embedding integration
        api_endpoint="http://172.17.0.3:11434",  # If using Docker you might need: http://host.docker.internal:11434
        model="nomic-embed-text:latest",  # The model to use
    ),
    generative_config=Configure.Generative.ollama(  # Configure the Ollama generative integration
        api_endpoint="http://172.17.0.3:11434",  # If using Docker you might need: http://host.docker.internal:11434
        model="llama3.2",  # The model to use
    ),
    properties=[
        Property(
            name="type",
            data_type=DataType.TEXT,
            skip_vectorization=False
        ),
        Property(
            name="page",
            data_type=DataType.INT,
            skip_vectorization=False
        ),
        Property(
            name="description",
            data_type=DataType.TEXT,
            skip_vectorization=False
        ),
        Property(
            name="text",
            data_type=DataType.TEXT,
            skip_vectorization=False
        ),
        Property(
            name="trace",
            data_type=DataType.TEXT,
            skip_vectorization=False
        ),
        Property(
            name="image",
            data_type=DataType.BLOB,
            skip_vectorization=True,  # Image won't be embedded
            index_null_state=True
        ),
    ]
)


client.close()

None


In [13]:
client = weaviate.connect_to_local(
    host="172.17.0.4",  
    port=8080,
    grpc_port=50051,
)

with open("/home/prime/Documents/multimodal-oran-rag/Output/O-RAN.WG3.TS.E2AP-R004-v07.00/O-RAN.WG3.TS.E2AP-R004-v07.00_cleaned.json", 'r') as f:
    data = json.load(f)

questions = client.collections.use("Grounded_nomic")

with questions.batch.fixed_size(batch_size=200) as batch:
    for d in data:
        properties = {
                "type": d["block_type"],
                "page": d["page"],
                "description": d["Description"],
                "text": d["Text"],
                "trace": d["Trace"],
            #    "Image": list(d["images"].values())[0] if d["images"] else None,
            }

        # Handle image properly
        if d["images"]:
            properties["image"] = d["images"]
       
        batch.add_object(properties)
        
        if batch.number_errors > 10:
            print("Batch import stopped due to excessive errors.")
            break

failed_objects = questions.batch.failed_objects
if failed_objects:
    print(f"Number of failed imports: {len(failed_objects)}")
    print(f"First failed object: {failed_objects[0]}")

client.close()  # Free up resources

In [1]:
import weaviate

client = weaviate.connect_to_local(
    host="172.17.0.4",  
    port=8080,
    grpc_port=50051,
)

questions = client.collections.get("Grounded_nomic")

# Get the schema
config = questions.config.get()

print("Current schema properties:")
for prop in config.properties:
    print(f"  - {prop.name}: {prop.data_type}")
    if hasattr(prop, 'skip_vectorization'):
        print(f"    skip_vectorization: {prop.skip_vectorization}")

client.close()

Current schema properties:
  - type: DataType.TEXT
  - page: DataType.INT
  - description: DataType.TEXT
  - text: DataType.TEXT
  - trace: DataType.TEXT
  - image: DataType.BLOB


In [2]:
import weaviate
import json
from weaviate.classes.query import Filter
client = weaviate.connect_to_local(
    host="172.17.0.4",  
    port=8080,
    grpc_port=50051,
)
questions = client.collections.use("Grounded_nomic")

my_filter = Filter.by_property("image").is_none(False)


response = questions.query.bm25(

    query=""" 

   The diagram shows interactions between Near-RT RICs and E2 Nodes:

"RIC SUBSCRIPTION MODIFICATION REQUIRED" from Near-RT RIC to E2 Node
"RIC SUBSCRIPTION MODIFICATION REFUSE" from E2 Node to Near-RT RIC
        """,
    limit=10
    
)

for obj in response.objects:
    print(json.dumps(obj.properties, indent=2))

client.close()  # Free up resources

{
  "type": "Figure",
  "trace": "8.2 RIC Functional procedures --> 8.2.6.2 Successful operation --> 8.2.6.3 Unsuccessful operation --> Figure 8.2.6.3-1: RIC Subscription Modification Required procedure, unsuccessful operation",
  "page": 29,
  "description": "The diagram shows interactions between Near-RT RICs and E2 Nodes:\n1. \"RIC SUBSCRIPTION MODIFICATION REQUIRED\" from Near-RT RIC to E2 Node\n2. \"RIC SUBSCRIPTION MODIFICATION REFUSE\" from E2 Node to Near-RT RIC",
  "text": "Figure 8.2.6.3-1: RIC Subscription Modification Required procedure, unsuccessful operation"
}
{
  "type": "Figure",
  "trace": "8.2 RIC Functional procedures --> 8.2.6.2 Successful operation --> Figure 8.2.6.2-1: RIC Subscription Modification Required procedure, successful operation",
  "page": 28,
  "description": "The diagram shows interactions between a Near-RT RIC and an E2 Node:\n1. \"RIC SUBSCRIPTION MODIFICATION REQUIRED\" from Near-RT RIC to E2 Node\n2. \"RIC SUBSCRIPTION MODIFICATION CONFIRM\" from

In [16]:
import weaviate

client = weaviate.connect_to_local(
    host="172.17.0.4",  
    port=8080,
    grpc_port=50051,
)

questions = client.collections.use("Grounded_nomic")

response = questions.generate.near_text(
    query="what is RIC service signaling",
    limit=3,
    grouped_task="Provided detailed answer to best of your ability, and cite source using trace."
)

print(response.generative.text)  # Inspect the generated text

client.close()  # Free up resources

Based on the provided information, it appears that the document is discussing the "RIC" (Radio Interface Control) functional procedures for a cellular network or telecommunications system.

The text mentions that multiple procedures use RIC Service signalling, including:

* RIC Indication procedure (Section 8.2.3)
* RIC Assistance Indication procedure (Section 8.2.12)
* RIC Service Load Status procedure (Section 8.2.8)

These procedures are all part of the RIC functional procedures, which are used to manage and control the communication between different network elements.

Unfortunately, the text does not provide detailed information on what each procedure entails or how it is implemented. However, based on the context, it appears that these procedures are related to the management of network resources, such as capacity allocation, handovers, and service load monitoring.

The source of this document is unclear, but based on the reference to RIC (Radio Interface Control) functional proc